# MDB Vegetation Vulnerability Workflow

This notebook orchestrates the vegetation vulnerability analysis.

## Steps:
1. Initialize Configuration and Spatial Data
2. Load and Process WIT Metrics
3. Load and Process TSLI (Time Since Last Inundation)
4. Load and Process NDVI
5. Load and Process Soil Moisture
6. Aggregate and Calculate Vulnerability Scores

# Background

Modified from the MDBA BWS Vulnerabilities Project

The BWS Priorities Project aimed to spatially and temporally summarise metrics of vulnerability (combining condition and stress) for vegetation and waterbirds in the Murray-Darling Basin with the aim of informing the setting of annual watering priorities for these target groups.

**Project report**: Hale, J., Brooks, S., Campbell, C. and McGinness, H. (2023) Assessing Vulnerability for use in Determining Basin-scale Environmental Watering Priorities. A Report to the Commonwealth Environmental Water Office, Canberra.

* [LInk to report from the DCCEEW website](https://www.dcceew.gov.au/sites/default/files/documents/assessing-vulnerability-use-determining-basin-scale-environmental-watering-priorities.pdf)

The Jupyter notebook is the final stage of data processing that pulls together multiple data sets to summarise and score the condition metrics, stress metrics and then add the scores to the final vulnerability metric.  Multiple input data files are read in, pivoted to tabular format with years as columns.  The measurement of vulnerability relies on first calculating the long-term baseline (mean of all years excluding the millennium drought) then scoring the deviation from the baseline.   Metrics calculated for ANAE ecosystem polygons are aggregated together as an area weighted average for larger spatial units (e.g. Ramsar sites, valleys).

## Data Inputs

 1. Australian National Aquatic Ecosystem (ANAE) mapping  v3 - The ANAE identifies different vegetation types and provides the spatial units used to summarise other data. Polygons < 1 Ha are removed as they are too small to meet the reliability requirements of the WIT tool and MODIS derived NDVI.  
 1. Geoscience Australia Wetland Insights Tool (WIT) - WIT data observations for all ANAE polygons > 1 Ha in the MDB 1986-present.  Raw data supplied by Geoscience Australia for individual observation dates through the Landsat Record summarised into daily, yearly, all-time and inundation event statistics (a separate jupyter notebook)
 1. Normalized Difference Vegetation Index (NDVI) - Average NDVI per ANAE polygon per year 1986-present calculated using google earth engine reducer: shared code: [(Flow-MER GoogleEarthEngine_scripts)](https://github.com/Flow-MER/GoogleEarthEngine_scripts)
 1. [Root Zone Soil Moisture (Australian Water Outlook)](https://awo.bom.gov.au/products/historical/soilMoisture-rootZone) - Mean root zone soil moisture per ANAE polygon per year was generated using ArcGIS but there are many ways to calculate the annual average per polygon from the AWO netcdf   
 1. Stress thresholds for vegetation based on durations since last inundation for different functional groups that were identified by experts are coded directly into this Jupyter Notebook

## Data Outputs

This notebook writes the various metric to the working directory in tabular format csv files (spatial units in rows, years in columns) that can be read by Microsoft Excel.  Baseline values and scores are added to the tables as additional columns. The output includes spatial scales that were not included in the BWS Vulnerabilities project report but may be useful for other investigations or to inform water planning at those locations (e.g. DIWA and Ramsar sites)

Output files for habitat metrics follow the naming convention: {metric}_{aggregator}_{year_window_width}yr_condition.csv
e.g.  pv_median_DIWA_5yr_condition.csv  is the median "pv" (green fractional cover) with ANAE polygons aggregated to larger DIWA wetland scales using a 5-year moving window in which to calculate rates of change.  

*NOTE:  The outputs generated from this notebook will vary from the previous work in the BWS Vulnerabilities report because:
1. removed the MDBA Stand Condition tool inputs
1. threshold NDVI inputs to positive values only (limits influence of areas of open water)
2. removed unvegetated ANAE classes (lakes, clay pans)


### Mapping the outputs

* Patterns can be visualised in GIS by joining the output files to the relevant spatial layers.  Many of the vegetation maps in the report used the ANAE polygons scale to visualise the patterns - this was done by joining **FINAL_BWSVulnerability_vegetation_ANAE.csv** to the **ANAEv3** using the **UID** polygon identifier.  Mapping whole Valley aggregated scores would be done by joining **FINAL_BWSVulnerability_vegetation_Valley.csv** to **BWSRegions.shp** using the **BWS_Region**.

## Processing Environment

Python 3.11.11

install requirements

```pip install -r requirements.txt```

## Repeating or extending the analysis to additional years of data

Extending the analysis requires:

1. collating new input data and appending to the current 1986-2024 source files
1. edit the definition of the **alltime** variable to extend past 2024.
1. re-run the notebook

Source data comes from a variety of places and requires a different technologies to assemble as outlined above.  The current source files should be used as the template to append to,  which should ensure the updated files will run with this workbook.  There is some additional code built into the workbook to re-build spatial relationships among data

The code was built to test the method within the confines of a project so it isn't always pretty.    If the logic is not clear please refer to the report and reach out to the report authors with questions.

***

## Contact

Dr Shane Brooks
<https://brooks.eco>

![Brooks.eco logo](brooks-logo.png "Brooks Ecology & Technology")


In [ ]:
import sys
import os

# Ensure local modules are found
sys.path.append(os.getcwd())

from config import veg_config
from data_loaders import (
    SpatialDataManager, 
    WitMetricsLoader, 
    TsliLoader, 
    NdviLoader, 
    SoilMoistureLoader
)
from processors import MetricProcessor, VulnerabilityAggregator

## 1. Initialize Spatial Data

In [ ]:
spatial_mgr = SpatialDataManager(veg_config)
spatial_mgr.load_data()

processor = MetricProcessor(veg_config, spatial_mgr)

## 2. Process WIT Metrics

In [ ]:
wit_loader = WitMetricsLoader(veg_config)
wit_df = wit_loader.load_data(valid_uids=spatial_mgr.anae_gdf["UID"])

# Process Vegetation Metrics (5yr window)
metrics = ["water+wet_median", "pv_median", "npv_median", "npv+pv+wet_median"]
processor.process_metric(
    wit_df, 
    metrics, 
    year_window_width=veg_config.VEG_WINDOW_WIDTH, 
    trend_window_width=veg_config.VEG_TREND_WIDTH
)

# Process Bare Soil (Reverse scores)
processor.process_metric(
    wit_df, 
    "bs_median", 
    year_window_width=veg_config.VEG_WINDOW_WIDTH, 
    trend_window_width=veg_config.VEG_TREND_WIDTH,
    reverse_scores=True
)

# Process Annual Metrics (1yr window)
metrics_1yr = ["water+wet_median", "npv+pv+wet_median", "pv_median"]
processor.process_metric(wit_df, metrics_1yr, year_window_width=1)

## 3. Process TSLI

In [ ]:
tsli_loader = TsliLoader(veg_config)
tsli_df = tsli_loader.load_data(spatial_mgr)

processor.process_tsli_scores(tsli_df)

## 4. Process NDVI

In [ ]:
ndvi_loader = NdviLoader(veg_config)
ndvi_df, ndvi_std_df = ndvi_loader.load_data(valid_uids=spatial_mgr.anae_gdf["UID"])

# Normalised NDVI
processor.process_metric(
    ndvi_df, 
    "NDVI", 
    year_window_width=veg_config.VEG_WINDOW_WIDTH, 
    trend_window_width=veg_config.VEG_TREND_WIDTH
)

# Standardised NDVI
processor.process_metric(
    ndvi_std_df, 
    "NDVI", 
    year_window_width=veg_config.VEG_WINDOW_WIDTH, 
    trend_window_width=veg_config.VEG_TREND_WIDTH,
    no_baseline=True,
    tag="_standardised"
)

# Annual NDVI
processor.process_metric(ndvi_df, "NDVI", year_window_width=1)

## 5. Process Soil Moisture

In [ ]:
sm_loader = SoilMoistureLoader(veg_config)
sm_df = sm_loader.load_data(valid_uids=spatial_mgr.anae_gdf["UID"])

# 5yr Window
processor.process_metric(
    sm_df, 
    "soilmoist", 
    year_window_width=veg_config.VEG_WINDOW_WIDTH, 
    trend_window_width=veg_config.VEG_TREND_WIDTH,
    _bins=[-1, 0.25, 0.5, 1],
    no_baseline=True
)

# Annual
processor.process_metric(
    sm_df, 
    "soilmoist", 
    year_window_width=1,
    _bins=[-1, 0.25, 0.5, 1],
    no_baseline=True
)

## 6. Calculate Final Vulnerability Scores

In [ ]:
aggregator = VulnerabilityAggregator(veg_config, spatial_mgr)
aggregator.combine_and_save()